# 01 · Ingest open data (PBH / BHTrans)

Downloads the crash datasets (2011+) from dados.pbh.gov.br, unifies drifting column names, fixes the 2011–2015 coordinate layout, flags default geocodes and writes parquet to `processed/`. See `docs/data_audit.md`.

In [1]:
# Setup: works on Colab (Drive + private repo) and locally (repo checkout).
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["BHTRAFFIC_DATA_ROOT"] = "/content/drive/MyDrive/transito_bh"
    token = userdata.get("GH_TOKEN")  # Colab Secrets: fine-grained token, read-only on the repo
    # Clone (not pip-from-git): the package resolves data/ and paper/ relative to the repo root.
    url = f"https://{token}@github.com/Dias0202/Transito_bh.git"
    if os.path.isdir("/content/Transito_bh"):
        !git -C /content/Transito_bh pull -q {url} main
    else:
        !git clone -q {url} /content/Transito_bh
        !git -C /content/Transito_bh remote set-url origin https://github.com/Dias0202/Transito_bh.git
    %pip -q install -e "/content/Transito_bh[geo,viz]"
    sys.path.insert(0, "/content/Transito_bh/src")
    os.chdir("/content/Transito_bh/notebooks")
else:
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd().parent / "src"))

In [2]:
from bhtraffic.ingest import pbh, sinistros
tables = sinistros.build()
{k: v.shape for k, v in tables.items()}

{'ocorrencias': (184350, 17), 'veiculos': (333111, 5), 'pessoas': (404957, 9)}

In [3]:
o = tables['ocorrencias']
print('years', o.year.min(), '-', o.year.max())
print('without coordinates:', round(o.lat.isna().mean(), 3))
print('default geocodes removed from map:', int(o.coord_placeholder.sum()))
o.groupby('year').size().rename('crashes').to_frame().T

years 2011 - 2024
without coordinates: 0.146
default geocodes removed from map: 6618


year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
crashes,16288,15140,14077,14907,13178,12473,12243,11656,13342,10624,11122,12148,13886,13266


Corridors sampled by the collector (`data/corridors.toml`): check the anchor points on a map.

In [4]:
from bhtraffic.geo import corridors
cs, bbs = corridors.load()
import pandas as pd
pd.DataFrame([{'id': c.id, 'name': c.name, 'area': c.area, 'points': len(c.path)} for c in cs])

,id,name,area,points
0,antonio_carlos,Av. Antônio Carlos,BH,3
1,cristiano_machado,Av. Cristiano Machado,BH,3
2,amazonas,Av. Amazonas,BH/Contagem,3
3,afonso_pena,Av. Afonso Pena,BH,2
4,raja_gabaglia,Av. Raja Gabaglia,BH,2
5,nossa_senhora_do_carmo,Av. N. Sra. do Carmo / BR-356,BH/Nova Lima,2
6,anel_sul,Anel Rodoviário (Olhos d'Água - Ivaí),BH,2
7,anel_norte,Anel Rodoviário (Ivaí - São Gabriel),BH,2
8,tereza_cristina,Av. Tereza Cristina (Barreiro),BH,3
9,vilarinho,Av. Vilarinho / Pedro I,BH,2
